# quaero_heads

Multi-head Gumbel subset-sampling model.

One head per target bin.  A shared GIN embeds all nodes; head $k$ maps the
embeddings of bin-$k$ nodes to logits and selects **exactly** $b_k$ of them
via relaxed Gumbel top-$k$ subset sampling (Xie & Ermon, 2019 — successive
softmax).  The per-bin quotas $b_k$ come from `find_sampling_plan`, so the
quota constraint is satisfied *by construction* — no penalty terms.

Training uses **edge-based batching**: the node set is fixed every step,
only the edge set is subsampled.

## Loss functions (three, selectable in the train loop)

**1. Soft hyperellipsoid at a chosen coverage level (active).**  Fit a
multivariate Gaussian to the selection via the membership-weighted empirical
mean and covariance.  The $q$-level ellipsoid is
$\{p : \mathrm{maha}^2(p) \le \chi^2_q(D)\}$, and the hard inside-test is
smoothed with a sigmoid of width $T_{\mathrm{ell}}$:

$$s_i = \sigma\!\left(\frac{\chi^2_q(D) - (p_i - \mu)^\top \Sigma^{-1} (p_i - \mu)}{T_{\mathrm{ell}}}\right)$$

Two symmetric penalty terms — non-members should not be inside, members
should not be outside:

$$\mathcal{L} = \frac{\sum_i (1 - x_i)\, s_i \;+\; \lambda_{\mathrm{out}} \sum_i x_i (1 - s_i)}{\sum_i x_i}$$

Minimising both drives the selection to coincide with its own level
ellipsoid.  A Gaussian fit puts $\approx (1-q)$ of its own mass outside the
$q$ ellipsoid, so the member-outside term has an irreducible floor and
pressures the selection to be more compact than Gaussian.  The weighted
moments are smooth in the selection weights and the chi-square threshold is
a constant, so the loss is fully differentiable.  No $|\Sigma|$ term appears
anywhere (volume-independence: inflating the covariance cannot lower the
penalty).  $\Sigma + \varepsilon I$ keeps the Cholesky stable.  Gradient
signal concentrates in the sigmoid band around the ellipsoid shell — widen
`ell_T` if training stalls.

**2. Soft hyperrectangle ratio.**  Maximise selected / nodes-inside-the-box:

$$\mathcal{L} = -\frac{\sum_i x_i}{\sum_i s_i}, \qquad
s_i = \prod_d \sigma\!\left(\tfrac{p_{i,d} - \mathrm{lo}_d}{T}\right)
\sigma\!\left(\tfrac{\mathrm{hi}_d - p_{i,d}}{T}\right)$$

The hard box is non-differentiable (min/max bounds over a thresholded
selection; step-function inside-count), so the bounds use membership-weighted
soft-min/max and the inside test uses per-dimension sigmoids.

**3. Negative soft internal edge count (auxiliary, default off)** —
$-\sum_{(i,j) \in E} x_i x_j$, enabled via `lam_edge > 0`.

In [ ]:
# !pip install pykeops torch_geometric plotly
# (pykeops needs CUDA; on CPU the L1 fallback in knn_descriptors is used)

In [ ]:
import pandas as pd, numpy as np, math
from scipy import stats as scipy_stats
from typing import Union, List, Tuple, Callable
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GINConv
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, remove_self_loops, subgraph as pyg_subgraph
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
def bin_target(df: pd.DataFrame, target_col: str) -> dict:
    """
    Discretise a target column using Scott's rule and return a mapping
    of bin_id -> list of row indices.

    Scott's rule:
        bin_width  h = 3.5 * std(x) * n^(-1/3)
        n_bins       = ceil((max(x) - min(x)) / h)
    """
    x = df[target_col].dropna().values
    n = len(x)

    if n < 2:
        raise ValueError("Need at least 2 non-null values to compute bins.")

    std = np.std(x, ddof=1)          # sample standard deviation
    if std == 0:
        raise ValueError("Target column has zero variance; cannot bin.")

    h       = 3.5 * std * (n ** (-1 / 3))          # Scott's bin width
    n_bins  = max(1, math.ceil((x.max() - x.min()) / h))

    bin_series = pd.cut(
        df[target_col],
        bins=n_bins,
        labels=range(n_bins),
        include_lowest=True
    )

    bin_dict = {}
    for bin_id, group in df.groupby(bin_series, observed=True):
        bin_dict[int(bin_id)] = list(group.index)

    return bin_dict, n_bins

In [ ]:
DIST_MAP = {
    "normal":      scipy_stats.norm,
    "gaussian":    scipy_stats.norm,
    "exponential": scipy_stats.expon,
    "beta":        scipy_stats.beta,
    "gamma":       scipy_stats.gamma,
    "uniform":     scipy_stats.uniform,
    "lognormal":   scipy_stats.lognorm,
}

def make_pdf(
    spec: Union[
        Tuple[str, dict],
        List[Tuple[str, dict, float]]
    ]
) -> Callable:
    """
    Single distribution:
        spec = ("normal", {"loc": 0, "scale": 1})

    Mixture (weights are auto-normalised):
        spec = [
            ("normal",      {"loc": 0, "scale": 1},  0.6),
            ("exponential", {"scale": 2},             0.4),
        ]
    """
    if isinstance(spec, tuple):
        name, params = spec
        dist = DIST_MAP[name.lower()]
        return lambda x: dist.pdf(x, **params)

    total_w    = sum(w for _, _, w in spec)
    components = [(DIST_MAP[n.lower()], p, w / total_w) for n, p, w in spec]

    def mixture_pdf(x):
        return sum(w * d.pdf(x, **p) for d, p, w in components)

    return mixture_pdf


def _bin_centers(df: pd.DataFrame, target_col: str, n_bins: int) -> np.ndarray:
    """Midpoint of each equal-width bin over the target column range."""
    lo, hi = df[target_col].min(), df[target_col].max()
    edges  = np.linspace(lo, hi, n_bins + 1)
    return (edges[:-1] + edges[1:]) / 2

In [ ]:
EPS = 1e-9   # prevents bins from collapsing to 0 when c = 0

def find_sampling_plan(
    bin_map:       dict,
    df:            pd.DataFrame,
    target_col:    str,
    n_bins:        int,
    pdf:           Callable,
    min_total_pct: float = 0.0,
) -> Tuple[dict, List[int], bool]:
    """
    Trough-anchored iteration: for c = 0, 1, ..., capacity(anchor bin),
    derive required counts for every bin as round((c + eps) * pdf_i /
    pdf_anchor).  Bins that were anchors in earlier (failed) rounds are
    clamped to their capacity and never block feasibility.  Picks the level
    whose total is the smallest value >= min_total_pct * len(df).

    Returns (required {bin_id: b_k}, peak_bins, feasible).
    """
    centers  = _bin_centers(df, target_col, n_bins)
    bin_ids  = sorted(bin_map.keys())
    pdf_vals = {bid: pdf(centers[bid]) for bid in bin_ids}

    peak_pdf  = max(pdf_vals.values())
    peak_bins = [bid for bid in bin_ids if np.isclose(pdf_vals[bid], peak_pdf)]

    min_total = round(min_total_pct * len(df))

    seen    = set()  # bins that were anchors in earlier, infeasible rounds
    anchors = sorted(bin_ids, key=lambda b: pdf_vals[b])

    for anchor in anchors:
        anchor_pdf = pdf_vals[anchor]
        anchor_cap = len(bin_map[anchor])

        feasible_levels = []
        for c in range(anchor_cap + 1):
            required = {}
            for bid in bin_ids:
                val = round((c + EPS) * pdf_vals[bid] / anchor_pdf)
                required[bid] = min(val, len(bin_map[bid])) if bid in seen else val
            if all(required[bid] <= len(bin_map[bid]) for bid in bin_ids):
                if sum(required.values()) >= min_total:
                    feasible_levels.append((sum(required.values()), required))

        if feasible_levels:
            _, best_required = min(feasible_levels, key=lambda x: x[0])
            return best_required, peak_bins, True

        seen.add(anchor)

    # every bin used as an anchor -- the only possible level is "take everything"
    required = {bid: len(bin_map[bid]) for bid in bin_ids}
    if sum(required.values()) >= min_total:
        return required, peak_bins, True
    return {}, peak_bins, False

In [ ]:
def knn_descriptors(pos: torch.Tensor, k: int):
    """kNN indices over descriptor coords using L1 distance.  Returns (N, k).

    Uses KeOps on GPU; falls back to a torch.cdist L1 computation on CPU so
    the model runs without CUDA (same L1 metric as the original).
    """
    N, D = pos.shape
    if pos.is_cuda:
        from pykeops.torch import LazyTensor
        x_i = LazyTensor(pos.view(N, 1, D))
        y_j = LazyTensor(pos.view(1, N, D))
        return (x_i - y_j).abs().sum(dim=-1).argKmin(K=k, dim=1)
    return torch.cdist(pos, pos, p=1).topk(k, largest=False).indices


def build_pyg_graph(feats, knn_geo):
    """
    feats   : (N, d)  node feature tensor
    knn_geo : (N, k)  each row lists k neighbor indices for that node
    """
    N, k = knn_geo.shape
    src = torch.arange(N, device=feats.device).unsqueeze(1).expand(N, k).reshape(-1)
    dst = knn_geo.reshape(-1)
    edge_index = torch.stack([src, dst], dim=0)            # (2, N*k)
    edge_index, _ = remove_self_loops(edge_index)
    edge_index = to_undirected(edge_index, num_nodes=N)    # adds reverse edges + dedupes
    return Data(x=feats, edge_index=edge_index)

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(1500),
    "feature_b": np.random.randn(1500),
    "feature_c": np.random.randn(1500),
    "target":    np.random.rand(1500),
})
target_col = "target"
desc_cols  = [c for c in df.columns if c != target_col]
N          = len(df)

bin_map, n_bins = bin_target(df, target_col)      # Scott's rule

# THE GIVEN DISTRIBUTION and the per-bin quotas b_k (model inputs)
pdf = make_pdf(("gaussian", {"loc": 0.5, "scale": 0.15}))
required, peak_bins, feasible = find_sampling_plan(
    bin_map, df, target_col, n_bins, pdf, min_total_pct=0.10,
)
print(f"feasible={feasible}  peak_bins={peak_bins}")
print(f"required: {required}")
print(f"total required: {sum(required.values())} / {N}")

# node histograms (target-bin one-hot) appended to descriptors as features
bin_ids_arr = pd.cut(df[target_col], bins=n_bins, labels=range(n_bins),
                     include_lowest=True).astype(int).to_numpy()
hists = torch.zeros((N, n_bins), device=device)
hists[torch.arange(N), torch.tensor(bin_ids_arr, device=device)] = 1.0

# kNN graph on DESCRIPTORS only
pos     = torch.tensor(df[desc_cols].to_numpy(), dtype=torch.float32, device=device)
knn_geo = knn_descriptors(pos, k=math.floor(math.sqrt(N)))
feats   = torch.cat([pos, hists], dim=1).to(torch.float32)

# per-bin node index tensors (fixed across edge batches)
bin_node_idx = {bid: torch.tensor(idxs, device=device, dtype=torch.long)
                for bid, idxs in bin_map.items()}

print(f"n_bins={n_bins}  feats={tuple(feats.shape)}  knn={tuple(knn_geo.shape)}")

In [ ]:
def make_mlp(*dims):
    """Linear -> ReLU -> ... -> Linear (no activation on last layer)."""
    layers = []
    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i + 1]))
        if i < len(dims) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


def gumbel_subset_sample(logits, k, tau=1.0, hard=False, eps=1e-20):
    """
    Relaxed top-k subset sampling (Xie & Ermon, 2019 -- successive softmax).

    logits : (n,) unnormalised scores for the n candidate nodes
    k      : number of items to select
    tau    : temperature (annealed during training)
    hard   : straight-through -- exact k-hot forward, soft gradients backward

    Returns x : (n,) soft k-hot vector with sum(x) = k, differentiable
    w.r.t. logits.  Gumbel noise is added once; each of the k rounds
    suppresses the mass already selected and takes a softmax, so gradients
    flow through every round.
    """
    g = -torch.log(-torch.log(torch.rand_like(logits) + eps) + eps)
    scores = (logits + g) / tau

    x = torch.zeros_like(logits)
    onehot_approx = torch.zeros_like(logits)
    for _ in range(k):
        mask   = torch.clamp(1.0 - onehot_approx, min=eps)   # suppress selected mass
        scores = scores + torch.log(mask)
        onehot_approx = F.softmax(scores, dim=0)
        x = x + onehot_approx

    if hard:
        idx    = torch.topk(x, k).indices
        x_hard = torch.zeros_like(x).scatter_(0, idx, 1.0)
        x = (x_hard - x).detach() + x
    return x

In [ ]:
class MultiHeadSubsetGIN(nn.Module):
    """
    One head per target bin.  A shared GIN embeds all nodes; head k maps
    embeddings of bin-k nodes to logits and Gumbel-samples exactly b_k of
    them.  Heads' soft selections are scattered back into a global (N,)
    selection vector, so sum(selection) = sum_k b_k by construction.
    """
    def __init__(self, in_channels, hidden_channels, num_gin_layers, n_bins):
        super().__init__()
        self.gin_layers = nn.ModuleList(
            [GINConv(make_mlp(in_channels, hidden_channels, hidden_channels), train_eps=True)]
        )
        for _ in range(num_gin_layers - 1):
            self.gin_layers.append(
                GINConv(make_mlp(hidden_channels, hidden_channels, hidden_channels), train_eps=True)
            )
        self.heads = nn.ModuleList([
            make_mlp(hidden_channels, hidden_channels, 1) for _ in range(n_bins)
        ])

    def forward(self, x, edge_index, bin_node_idx, required, tau=1.0, hard=False):
        """
        x            : (N, in_channels)  full node feature tensor (nodes fixed)
        edge_index   : (2, E_batch)      current edge batch
        bin_node_idx : {bin_id: LongTensor of node indices in that bin}
        required     : {bin_id: b_k}     per-bin quotas (model inputs)

        Returns selection (N,), logits_all (N,)
        """
        h = x
        for gin in self.gin_layers:
            h = F.relu(gin(h, edge_index))

        selection  = torch.zeros(x.size(0), device=x.device)
        logits_all = torch.zeros(x.size(0), device=x.device)

        for bid, idx in bin_node_idx.items():
            b_k = int(required.get(bid, 0))
            if b_k == 0 or idx.numel() == 0:
                continue
            logits = self.heads[bid](h[idx]).squeeze(-1)      # (n_k,)
            logits_all[idx] = logits
            x_k = gumbel_subset_sample(logits, min(b_k, idx.numel()), tau=tau, hard=hard)
            selection = selection.index_add(0, idx, x_k)       # scatter to global positions

        return selection, logits_all

In [ ]:
def negative_internal_edge_loss(selection, edge_index):
    """
    -sum_{(i,j) in E} x_i * x_j  -- maximise the soft internal edge count.
    On an undirected PyG graph each edge appears in both directions
    (constant factor 2, harmless for optimisation).
    """
    row, col = edge_index
    return -(selection[row] * selection[col]).sum()


def soft_hyperrectangle_ratio_loss(selection, pos, T=0.1, eps=1e-9):
    """
    Differentiable surrogate for:  -(#selected) / (#nodes inside the bounding
    hyperrectangle of the selected nodes).

    The hard version is non-differentiable in two places: (1) the box bounds
    (min/max over a thresholded selection) and (2) the inside-count (a step
    function).  Both are smoothed:

      bounds : membership-weighted soft-min/max -- softmax over coordinates
               with log(selection) added, so only selected nodes shape the
               box; -> hard min/max as T -> 0
      count  : soft inside-indicator, product over dimensions of
               sigmoid((p - lo)/T) * sigmoid((hi - p)/T)

    selection : (N,) soft selection values in [0, 1]
    pos       : (N, D) descriptor coordinates
    """
    log_m = torch.log(selection + eps)                     # (N,)

    # soft box bounds, per dimension
    w_hi = F.softmax( pos / T + log_m.unsqueeze(1), dim=0) # (N, D)
    w_lo = F.softmax(-pos / T + log_m.unsqueeze(1), dim=0)
    hi   = (w_hi * pos).sum(0)                             # (D,)
    lo   = (w_lo * pos).sum(0)

    # soft inside-count
    inside = (torch.sigmoid((pos - lo) / T)
              * torch.sigmoid((hi - pos) / T)).prod(dim=1) # (N,)

    n_selected = selection.sum()
    n_inside   = inside.sum() + eps

    ratio = n_selected / n_inside
    return -ratio, ratio.item(), n_inside.item()


def soft_hyperellipsoid_loss(selection, pos, level=0.75, ell_T=0.5,
                             lam_out=1.0, eps_reg=1e-4, eps=1e-9):
    """
    Fit a multivariate Gaussian to the (softly) selected nodes via the
    membership-weighted empirical mean and covariance; the q-level ellipsoid
    is {p : maha^2(p) <= chi2_q(D)}, its hard inside-test smoothed by a
    sigmoid of width ell_T:

        s_i = sigmoid((m2_thr - maha_i^2) / ell_T)

    Two symmetric penalty terms:
      non-members INSIDE the ellipsoid : (1 - x_i) * s_i
      members OUTSIDE the ellipsoid    : x_i * (1 - s_i), weighted by lam_out

    Minimising both drives the selection to coincide with its own level
    ellipsoid: no bystanders swallowed, no members straggling outside.
    Note a Gaussian fit puts ~(1 - level) of its own mass outside the level
    ellipsoid, so the member-outside term has an irreducible floor and
    pressures the selection to be more compact than Gaussian.

    Fully differentiable: weighted moments are smooth in the selection
    weights, the chi-square threshold is a constant, and no |Sigma| term
    appears anywhere (volume-independence: inflating the covariance cannot
    lower the penalty).

    selection : (N,) soft selection values in [0, 1]
    pos       : (N, D) descriptor coordinates
    """
    D = pos.size(1)
    m2_thr = float(scipy_stats.chi2.ppf(level, df=D))

    w  = selection / (selection.sum() + eps)                     # (N,)
    mu = (w.unsqueeze(1) * pos).sum(0)                           # (D,)

    diff  = pos - mu                                             # (N, D)
    Sigma = (w.unsqueeze(1) * diff).T @ diff                     # (D, D)
    Sigma = Sigma + eps_reg * torch.eye(D, device=pos.device)

    L    = torch.linalg.cholesky(Sigma)                          # stable inverse
    z    = torch.linalg.solve_triangular(L, diff.T, upper=False) # (D, N)
    maha = (z ** 2).sum(0)                                       # (N,) squared Mahalanobis

    s = torch.sigmoid((m2_thr - maha) / ell_T)                   # soft inside-the-q-ellipsoid

    non_member_mass = ((1.0 - selection) * s).sum()          # bystanders inside
    member_out_mass = (selection * (1.0 - s)).sum()          # members outside

    loss = (non_member_mass + lam_out * member_out_mass) / (selection.sum() + eps)
    return loss, non_member_mass.item(), member_out_mass.item()

In [ ]:
# Model
in_channels     = feats.shape[1]
hidden_channels = 256
num_gin_layers  = 3

# Training
lr              = 1e-4
num_epochs      = 250
edge_batch_size = 8192
ell_level       = 0.75   # ellipsoid coverage level for the loss (0.5, 0.75, 0.95, ...)
ell_T           = 0.5    # sigmoid width of the soft ellipsoid boundary
ell_reg         = 1e-4   # covariance ridge for the hyperellipsoid loss
lam_out         = 1.0    # weight of the members-outside-the-ellipsoid penalty
box_T           = 0.1    # temperature of the soft hyperrectangle (bounds + inside test)
lam_edge        = 0.0    # optional auxiliary internal-edge loss weight (0 = off)

# Gumbel-Softmax temperature annealing
tau_start       = 1.0
tau_min         = 0.1
anneal_rate     = 0.01   # tau = max(tau_min, tau_start * exp(-anneal_rate * epoch))

In [ ]:
data = build_pyg_graph(feats, knn_geo).to(device)
E    = data.edge_index.size(1)
print(f"graph: {data.num_nodes} nodes, {E} directed edges")

model = MultiHeadSubsetGIN(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    num_gin_layers=num_gin_layers,
    n_bins=n_bins,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
for epoch in range(num_epochs):
    tau = max(tau_min, tau_start * math.exp(-anneal_rate * epoch))
    model.train()
    epoch_loss = 0.0

    # edge-based batching: nodes fixed, edges shuffled and split
    perm = torch.randperm(E, device=device)
    n_batches = 0
    for s in range(0, E, edge_batch_size):
        batch_edges = data.edge_index[:, perm[s:s + edge_batch_size]]
        optimizer.zero_grad()

        selection, _ = model(data.x, batch_edges, bin_node_idx, required, tau=tau)

        # main objective: non-members inside + members outside the ell_level ellipsoid
        loss, nm_mass, mo_mass = soft_hyperellipsoid_loss(
            selection, pos, level=ell_level, ell_T=ell_T,
            lam_out=lam_out, eps_reg=ell_reg,
        )
        # alternative: soft selected/inside-hyperrectangle ratio
        # loss, ratio, n_inside = soft_hyperrectangle_ratio_loss(selection, pos, T=box_T)
        if lam_edge > 0:
            loss = loss + lam_edge * negative_internal_edge_loss(selection, batch_edges)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches  += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"epoch {epoch + 1}/{num_epochs}  tau={tau:.3f}  "
              f"avg_loss={epoch_loss / n_batches:.4f}  "
              f"non_member_mass={nm_mass:.1f}  member_out_mass={mo_mass:.1f}")

In [ ]:
# Inference: hard straight-through sampling on the FULL edge set --
# exactly b_k nodes per bin by construction.
model.eval()
with torch.no_grad():
    selection, logits_all = model(
        data.x, data.edge_index, bin_node_idx, required, tau=tau_min, hard=True,
    )

selected_nodes = (selection > 0.5).nonzero(as_tuple=True)[0]
internal_edges = int(
    (selection[data.edge_index[0]] * selection[data.edge_index[1]]).sum().item() / 2
)
print(f"{selected_nodes.numel()} nodes selected "
      f"(total required: {sum(required.values())})")
print(f"internal (subset-internal) edges: {internal_edges}")

In [ ]:
# Hard hyperrectangle ratio (exact, non-differentiable -- fine at eval time)
sel_pos    = pos[selected_nodes]
lo_h, hi_h = sel_pos.min(0).values, sel_pos.max(0).values
in_box     = ((pos >= lo_h) & (pos <= hi_h)).all(dim=1)
hard_ratio = selected_nodes.numel() / int(in_box.sum())
print(f"bounding hyperrectangle: lo={lo_h.cpu().numpy().round(3)}  "
      f"hi={hi_h.cpu().numpy().round(3)}")
print(f"hard ratio: {selected_nodes.numel()}/{int(in_box.sum())} = {hard_ratio:.3f}")

In [ ]:
# Hard hyperellipsoid check (exact, non-differentiable -- fine at eval time):
# fit the Gaussian to the hard-selected nodes, count non-members inside the
# ell_level ellipsoid (Mahalanobis^2 <= chi2_{ell_level}(D)) -- the same
# ellipsoid the training loss targets.
D_dim   = pos.size(1)
sel_pos = pos[selected_nodes]

mu_h    = sel_pos.mean(0)
diff_h  = sel_pos - mu_h
Sigma_h = (diff_h.T @ diff_h) / sel_pos.size(0)
Sigma_h = Sigma_h + ell_reg * torch.eye(D_dim, device=pos.device)

L_h    = torch.linalg.cholesky(Sigma_h)
z_all  = torch.linalg.solve_triangular(L_h, (pos - mu_h).T, upper=False)
maha_all = (z_all ** 2).sum(0)                         # (N,)

m2_thr    = scipy_stats.chi2.ppf(ell_level, df=D_dim)
inside    = maha_all <= m2_thr
is_member = torch.zeros_like(inside)
is_member[selected_nodes] = True

nm_inside = int((inside & ~is_member).sum())
m_inside  = int((inside & is_member).sum())
print(f"{ell_level:.0%} ellipsoid (chi2={m2_thr:.2f}, D={D_dim}): "
      f"{m_inside}/{selected_nodes.numel()} members inside, "
      f"{nm_inside} non-members inside")
print(f"member share inside ellipsoid: "
      f"{m_inside / max(m_inside + nm_inside, 1):.3f}")

In [ ]:
# Per-bin selected vs required -- the quota is met by construction
sel_np = selection.cpu().numpy()
rows = []
for bid in sorted(bin_map.keys()):
    idx      = bin_node_idx[bid].cpu().numpy()
    selected = int((sel_np[idx] > 0.5).sum())
    rows.append({
        "bin_id":    bid,
        "available": len(idx),
        "required":  required.get(bid, 0),
        "selected":  selected,
        "met":       selected == required.get(bid, 0),
    })
report = pd.DataFrame(rows)
report.loc["TOTAL"] = ["TOTAL", report["available"].sum(),
                       report["required"].sum(), report["selected"].sum(),
                       bool(report["met"].all())]
display(report)

In [ ]:
def count_induced_components(subgroup_nodes, edge_index, num_nodes):
    """Number of connected components of the subgraph induced by the selection."""
    n = subgroup_nodes.size(0)
    if n == 0:
        return 0

    sub_edge_index, _ = pyg_subgraph(
        subgroup_nodes, edge_index,
        relabel_nodes=True, num_nodes=num_nodes,
    )

    if sub_edge_index.size(1) == 0:
        return n   # all nodes are isolated

    row = sub_edge_index[0].cpu().numpy()
    col = sub_edge_index[1].cpu().numpy()
    adj = csr_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
    n_components, _ = connected_components(adj, directed=False, connection='weak')
    return n_components


n_comp = count_induced_components(
    selected_nodes.cpu(), data.edge_index.cpu(), data.num_nodes,
)
print(f"selected subset: {selected_nodes.numel()} nodes, "
      f"{n_comp} connected component(s) in the descriptor kNN graph")

In [ ]:
# Target-distribution check: subset histogram vs the specified distribution
selected_idx_np = selected_nodes.cpu().numpy()
sel_targets     = df.loc[selected_idx_np, target_col]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df[target_col], bins=n_bins, alpha=0.35, density=True,
        color="steelblue", label="full dataset")
ax.hist(sel_targets, bins=n_bins, alpha=0.75, density=True,
        color="tomato", label=f"subset (n={len(sel_targets)})")

# overlay the specified pdf
xs = np.linspace(df[target_col].min(), df[target_col].max(), 300)
ax.plot(xs, [pdf(x) for x in xs], "k--", lw=1.5, label="specified pdf")

ax.set_title("Multi-head Gumbel subset — target distribution")
ax.set_xlabel(target_col)
ax.set_ylabel("density")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive 3D view of the selected subset in descriptor space
import plotly.graph_objects as go

sel_mask = df.index.isin(set(selected_idx_np.tolist()))
cols     = desc_cols

fig = go.Figure()

if len(cols) >= 3:
    fig.add_trace(go.Scatter3d(
        x=df.loc[~sel_mask, cols[0]], y=df.loc[~sel_mask, cols[1]], z=df.loc[~sel_mask, cols[2]],
        mode="markers",
        marker=dict(size=2, color="steelblue", opacity=0.15),
        name="unselected",
    ))
    fig.add_trace(go.Scatter3d(
        x=df.loc[sel_mask, cols[0]], y=df.loc[sel_mask, cols[1]], z=df.loc[sel_mask, cols[2]],
        mode="markers",
        marker=dict(size=3, color="tomato", opacity=0.7),
        name=f"selected (n={sel_mask.sum()})",
    ))
    fig.update_layout(
        scene=dict(xaxis_title=cols[0], yaxis_title=cols[1], zaxis_title=cols[2]),
    )
else:
    fig.add_trace(go.Scatter(
        x=df.loc[~sel_mask, cols[0]], y=df.loc[~sel_mask, cols[1]],
        mode="markers",
        marker=dict(size=4, color="steelblue", opacity=0.15),
        name="unselected",
    ))
    fig.add_trace(go.Scatter(
        x=df.loc[sel_mask, cols[0]], y=df.loc[sel_mask, cols[1]],
        mode="markers",
        marker=dict(size=5, color="tomato", opacity=0.7),
        name=f"selected (n={sel_mask.sum()})",
    ))
    fig.update_layout(xaxis_title=cols[0], yaxis_title=cols[1])

fig.update_layout(
    title=f"Multi-head Gumbel subset  ({n_comp} connected components, "
          f"{internal_edges} internal edges)",
    width=800, height=650,
    legend=dict(x=0.02, y=0.98),
)

fig.show()